# Version C Full Experiment: Hybrid Recommendation (C0–C4)

Version C combines **collaborative filtering** (from Version B) and **content-based recommendation** (from Version A) into hybrid models.

| Model | Description |
|---|---|
| C0 | SVD collaborative filtering baseline (same as B4) |
| C1 | TF-IDF user-profile content baseline (same as A3) |
| C2 | Weighted hybrid: normalized CF + content scores |
| C3 | Switching hybrid: CF for rich-history users, content for sparse users |
| C4 | Rank fusion hybrid: reciprocal rank fusion of CF and content lists |

First run `DEBUG_MODE=True` to verify correctness.
Then set `DEBUG_MODE=False` and `FULL_RUN=True` for final results.

In [ ]:
import os
import json
import time
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

In [ ]:
DATA_DIR = "../../Data/Pure_Data"
RESULTS_DIR = "../Results"
FIGURES_DIR = "../Results/Figures"

DEBUG_MODE = True
MAX_DEBUG_USERS = 1000
FULL_RUN = False

POSITIVE_THRESHOLD = 4
K = 10
RANDOM_STATE = 42

# C0 SVD tuning
C0_COMPONENT_VALUES = [32, 64, 128]
# C1 TF-IDF tuning
C1_MAX_FEATURES_VALUES = [5000, 10000, 20000]
# C2 weighted hybrid alpha (CF weight; content weight = 1 - alpha)
C2_ALPHA_VALUES = [0.3, 0.5, 0.7, 0.9]
# C3 switching threshold (min train interactions to use CF)
C3_THRESHOLD_VALUES = [5, 10, 20]
# C4 rank fusion constant
C4_RRF_K_VALUES = [10, 60]

if DEBUG_MODE:
    C0_COMPONENT_VALUES = [32]
    C1_MAX_FEATURES_VALUES = [5000]
    C2_ALPHA_VALUES = [0.5, 0.7]
    C3_THRESHOLD_VALUES = [10]
    C4_RRF_K_VALUES = [60]

np.random.seed(RANDOM_STATE)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("DEBUG_MODE:", DEBUG_MODE)
print("FULL_RUN:", FULL_RUN)

In [ ]:
# ---- Load data ----
required_files = [
    "interactions_filtered.csv",
    "interactions_train_filtered.csv",
    "interactions_test_filtered.csv",
    "recipe_model_table.csv",
]

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

for f in required_files:
    p = os.path.join(DATA_DIR, f)
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required file: {p}")

interactions_filtered = pd.read_csv(os.path.join(DATA_DIR, "interactions_filtered.csv"))
train = pd.read_csv(os.path.join(DATA_DIR, "interactions_train_filtered.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "interactions_test_filtered.csv"))
recipe_model_table = pd.read_csv(os.path.join(DATA_DIR, "recipe_model_table.csv"))

for name, df in [("train", train), ("test", test)]:
    for col in ["user_id", "recipe_id", "rating"]:
        if col not in df.columns:
            raise ValueError(f"{name} missing required column: {col}")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("recipe_model_table shape:", recipe_model_table.shape)
display(train.head())

In [ ]:
# ---- Build user sets and eval users ----
test_positive = test[test["rating"] >= POSITIVE_THRESHOLD].copy()

def build_user_sets(df):
    out = defaultdict(set)
    for u, i in zip(df["user_id"].values, df["recipe_id"].values):
        out[u].add(i)
    return out

user_train_all_items = build_user_sets(train)
user_train_positive_items = build_user_sets(train[train["rating"] >= POSITIVE_THRESHOLD])
user_test_relevant_items = build_user_sets(test_positive)

# Count train interactions per user (for switching hybrid)
user_train_count = train.groupby("user_id").size().to_dict()

eval_users = sorted(user_test_relevant_items.keys())
if DEBUG_MODE:
    eval_users = eval_users[:MAX_DEBUG_USERS]
    user_test_relevant_items = {u: user_test_relevant_items[u] for u in eval_users}

print("Relevant test users for evaluation:", len(eval_users))
print("Positive threshold:", POSITIVE_THRESHOLD)

In [ ]:
# ---- Evaluation functions (same protocol as A and B) ----
def precision_at_k(rec, rel, k):
    rec = rec[:k]
    if k == 0 or len(rec) == 0:
        return 0.0
    return sum(1 for x in rec if x in rel) / k

def recall_at_k(rec, rel, k):
    if len(rel) == 0:
        return 0.0
    rec = rec[:k]
    return sum(1 for x in rec if x in rel) / len(rel)

def ndcg_at_k(rec, rel, k):
    rec = rec[:k]
    if len(rec) == 0 or len(rel) == 0:
        return 0.0
    dcg = sum(1.0 / np.log2(rank + 1) for rank, item in enumerate(rec, start=1) if item in rel)
    ideal = min(len(rel), k)
    idcg = sum(1.0 / np.log2(r + 1) for r in range(1, ideal + 1)) if ideal > 0 else 0.0
    return dcg / idcg if idcg > 0 else 0.0

def hit_at_k(rec, rel, k):
    return 1.0 if any(x in rel for x in rec[:k]) else 0.0

def evaluate_model(model_id, model_name, phase, split_name, recommendations, runtime_seconds):
    per_user_rows = []
    unique_rec = set()
    total_recs = 0
    for u in eval_users:
        rel = user_test_relevant_items.get(u, set())
        rec_items = recommendations.get(u, [])
        rec_ids = [x[0] if isinstance(x, tuple) else x for x in rec_items][:K]
        unique_rec.update(rec_ids)
        total_recs += len(rec_ids)
        p = precision_at_k(rec_ids, rel, K)
        r = recall_at_k(rec_ids, rel, K)
        n = ndcg_at_k(rec_ids, rel, K)
        h = hit_at_k(rec_ids, rel, K)
        per_user_rows.append({"user_id": u, "model_id": model_id,
                              "precision_at_k": p, "recall_at_k": r,
                              "ndcg_at_k": n, "hit_at_k": h,
                              "num_relevant_items": len(rel),
                              "num_recommended_items": len(rec_ids)})
    per_user_df = pd.DataFrame(per_user_rows)
    eval_count = len(per_user_df)
    catalog_cov = len(unique_rec) / train["recipe_id"].nunique() if train["recipe_id"].nunique() > 0 else 0.0
    sec_per = runtime_seconds / eval_count if eval_count > 0 else np.nan
    agg = {
        "version": "C", "phase": phase, "model_id": model_id,
        "model_name": model_name, "split": split_name, "k": K,
        "evaluated_users": int(eval_count),
        "precision_at_k": float(per_user_df["precision_at_k"].mean()),
        "recall_at_k": float(per_user_df["recall_at_k"].mean()),
        "ndcg_at_k": float(per_user_df["ndcg_at_k"].mean()),
        "hit_at_k": float(per_user_df["hit_at_k"].mean()),
        "catalog_coverage_at_k": float(catalog_cov),
        "total_recommendations": int(total_recs),
        "unique_recommended_items": int(len(unique_rec)),
        "runtime_seconds": float(runtime_seconds),
        "seconds_per_user": float(sec_per) if not np.isnan(sec_per) else np.nan,
    }
    return agg, per_user_df

In [ ]:
# ---- Popularity fallback (same as B0) ----
pop_global_mean = train["rating"].mean()
pop = train.groupby("recipe_id").agg(R=("rating", "mean"), v=("rating", "count")).reset_index()
pop["score"] = (pop["v"] / (pop["v"] + 100)) * pop["R"] + (100 / (pop["v"] + 100)) * pop_global_mean
pop = pop.sort_values("score", ascending=False)
pop_ranked = list(zip(pop["recipe_id"].tolist(), pop["score"].tolist()))

def popularity_recs_for_users(pop_ranked, users):
    out = {}
    for u in users:
        seen = user_train_all_items.get(u, set())
        recs = []
        for rid, score in pop_ranked:
            if rid in seen:
                continue
            recs.append((rid, float(score)))
            if len(recs) >= K:
                break
        out[u] = recs
    return out

print("Popularity fallback ready. Top recipe score:", pop_ranked[0][1])

## C0: SVD Collaborative Filtering Baseline

Same as Version B's B4. This gives us the CF component for hybridization.

In [ ]:
# ---- C0: SVD CF ----
def build_svd_model(train_df, n_components):
    """Build SVD model and return score matrix + mappings."""
    users = np.sort(train_df["user_id"].unique())
    items = np.sort(train_df["recipe_id"].unique())
    u2i = {u: idx for idx, u in enumerate(users)}
    r2i = {r: idx for idx, r in enumerate(items)}
    i2r = {idx: r for r, idx in r2i.items()}
    rows = train_df["user_id"].map(u2i).values
    cols = train_df["recipe_id"].map(r2i).values
    vals = train_df["rating"].astype(float).values
    mat = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)), dtype=np.float32)
    n_comp = min(n_components, min(mat.shape) - 1) if min(mat.shape) > 1 else 1
    svd = TruncatedSVD(n_components=max(1, n_comp), random_state=RANDOM_STATE)
    user_emb = svd.fit_transform(mat)
    item_emb = svd.components_.T
    score_mat = user_emb @ item_emb.T
    return score_mat, u2i, r2i, i2r

def svd_score_for_user(u, score_mat, u2i, i2r):
    """Return {recipe_id: score} dict for a user."""
    if u not in u2i:
        return {}
    uidx = u2i[u]
    scores = score_mat[uidx]
    return {i2r[idx]: float(scores[idx]) for idx in range(len(scores))}

def svd_recommendations(score_mat, u2i, i2r, users_eval):
    recs = {}
    for u in users_eval:
        if u not in u2i:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue
        uidx = u2i[u]
        seen = user_train_all_items.get(u, set())
        s = score_mat[uidx]
        idx_rank = np.argsort(-s)
        out = []
        for idx in idx_rank:
            item = i2r[int(idx)]
            if item in seen:
                continue
            out.append((item, float(s[idx])))
            if len(out) >= K:
                break
        if not out:
            out = popularity_recs_for_users(pop_ranked, [u])[u]
        recs[u] = out
    return recs

metrics_rows = []
tuning_rows = []
per_user_frames = []
runtime_rows = []
split_name = "filtered_temporal"

c0_best = None
for comp in tqdm(C0_COMPONENT_VALUES, desc="C0 SVD tuning"):
    t0 = time.time()
    score_mat, u2i_svd, r2i_svd, i2r_svd = build_svd_model(train, comp)
    recs = svd_recommendations(score_mat, u2i_svd, i2r_svd, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"C0_svd{comp}", f"SVD CF baseline {comp}", "C0", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"n_components": comp})
    agg["notes"] = "SVD collaborative filtering baseline (same method as B4)"
    tuning_rows.append(agg.copy())
    if (c0_best is None) or (agg["ndcg_at_k"] > c0_best["agg"]["ndcg_at_k"]):
        c0_best = {"agg": agg, "per_user": per_user, "recs": recs,
                   "score_mat": score_mat, "u2i": u2i_svd, "r2i": r2i_svd, "i2r": i2r_svd}

metrics_rows.append(c0_best["agg"])
per_user_frames.append(c0_best["per_user"])
runtime_rows.append({"phase": "C0", "model_id": c0_best["agg"]["model_id"],
                      "runtime_seconds": c0_best["agg"]["runtime_seconds"],
                      "evaluated_users": c0_best["agg"]["evaluated_users"],
                      "seconds_per_user": c0_best["agg"]["seconds_per_user"]})

print(f"C0 best: {c0_best['agg']['model_id']}  NDCG@{K}: {c0_best['agg']['ndcg_at_k']:.6f}")

## C1: TF-IDF Content-Based User Profile Baseline

Same approach as Version A's A3. Builds a user preference vector from liked recipes' TF-IDF features.

In [ ]:
# ---- C1: TF-IDF content ----
# Build combined text field for each recipe
text_cols = []
for col in ["name", "description", "tags_text", "ingredients_text"]:
    if col in recipe_model_table.columns:
        text_cols.append(col)

if not text_cols:
    # Fallback: use any text-like columns
    for col in recipe_model_table.columns:
        if recipe_model_table[col].dtype == object:
            text_cols.append(col)
            if len(text_cols) >= 4:
                break

print("Text columns for TF-IDF:", text_cols)

# Detect recipe id column name (may be "id" or "recipe_id")
recipe_id_col = "recipe_id" if "recipe_id" in recipe_model_table.columns else "id"
print("Recipe ID column:", recipe_id_col)

recipe_model_table["combined_text_v"] = recipe_model_table[text_cols].fillna("").agg(" ".join, axis=1)
recipe_id_to_idx = {rid: idx for idx, rid in enumerate(recipe_model_table[recipe_id_col].values)}
recipe_ids_array = recipe_model_table[recipe_id_col].values

def build_tfidf_model(max_features):
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words="english", sublinear_tf=True)
    tfidf_matrix = vectorizer.fit_transform(recipe_model_table["combined_text_v"])
    return vectorizer, tfidf_matrix

def content_user_profile_recs(tfidf_matrix, users_eval):
    """Build user profile from mean TF-IDF of liked recipes, then rank by cosine similarity."""
    from sklearn.metrics.pairwise import cosine_similarity
    recs = {}
    train_recipes_in_model = set(recipe_id_to_idx.keys())
    for u in users_eval:
        liked = user_train_positive_items.get(u, set())
        liked_in_model = [recipe_id_to_idx[r] for r in liked if r in recipe_id_to_idx]
        if not liked_in_model:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue
        user_vec = tfidf_matrix[liked_in_model].mean(axis=0)
        user_vec = np.asarray(user_vec).reshape(1, -1)
        sims = cosine_similarity(user_vec, tfidf_matrix).flatten()
        seen = user_train_all_items.get(u, set())
        idx_rank = np.argsort(-sims)
        out = []
        for idx in idx_rank:
            rid = int(recipe_ids_array[idx])
            if rid in seen:
                continue
            out.append((rid, float(sims[idx])))
            if len(out) >= K:
                break
        if not out:
            out = popularity_recs_for_users(pop_ranked, [u])[u]
        recs[u] = out
    return recs

def content_score_for_user(u, tfidf_matrix):
    """Return {recipe_id: similarity_score} dict for a user."""
    from sklearn.metrics.pairwise import cosine_similarity
    liked = user_train_positive_items.get(u, set())
    liked_in_model = [recipe_id_to_idx[r] for r in liked if r in recipe_id_to_idx]
    if not liked_in_model:
        return {}
    user_vec = tfidf_matrix[liked_in_model].mean(axis=0)
    user_vec = np.asarray(user_vec).reshape(1, -1)
    sims = cosine_similarity(user_vec, tfidf_matrix).flatten()
    return {int(recipe_ids_array[idx]): float(sims[idx]) for idx in range(len(sims))}

c1_best = None
best_tfidf_matrix = None
for mf in tqdm(C1_MAX_FEATURES_VALUES, desc="C1 TF-IDF tuning"):
    t0 = time.time()
    _, tfidf_mat = build_tfidf_model(mf)
    recs = content_user_profile_recs(tfidf_mat, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"C1_tfidf{mf}", f"TF-IDF content baseline {mf}", "C1", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"max_features": mf})
    agg["notes"] = "TF-IDF user-profile content baseline (same method as A3)"
    tuning_rows.append(agg.copy())
    if (c1_best is None) or (agg["ndcg_at_k"] > c1_best["agg"]["ndcg_at_k"]):
        c1_best = {"agg": agg, "per_user": per_user, "recs": recs, "tfidf_matrix": tfidf_mat}

metrics_rows.append(c1_best["agg"])
per_user_frames.append(c1_best["per_user"])
runtime_rows.append({"phase": "C1", "model_id": c1_best["agg"]["model_id"],
                      "runtime_seconds": c1_best["agg"]["runtime_seconds"],
                      "evaluated_users": c1_best["agg"]["evaluated_users"],
                      "seconds_per_user": c1_best["agg"]["seconds_per_user"]})
best_tfidf_matrix = c1_best["tfidf_matrix"]

print(f"C1 best: {c1_best['agg']['model_id']}  NDCG@{K}: {c1_best['agg']['ndcg_at_k']:.6f}")

## C2: Weighted Hybrid

Combine CF and content scores with a tunable weight `alpha`:

```
hybrid_score = alpha * norm(cf_score) + (1 - alpha) * norm(content_score)
```

Both scores are min-max normalized to [0, 1] before combining.

In [ ]:
# ---- C2: Weighted hybrid ----
def weighted_hybrid_recs(alpha, score_mat_svd, u2i_svd, i2r_svd, tfidf_matrix, users_eval):
    recs = {}
    for u in users_eval:
        seen = user_train_all_items.get(u, set())
        cf_scores = svd_score_for_user(u, score_mat_svd, u2i_svd, i2r_svd)
        ct_scores = content_score_for_user(u, tfidf_matrix)

        # Collect all candidate recipe ids (union of CF and content)
        all_items = set(cf_scores.keys()) | set(ct_scores.keys())
        all_items -= seen

        if not all_items:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue

        # Normalize CF scores
        cf_vals = np.array([cf_scores.get(r, 0.0) for r in all_items])
        cf_min, cf_max = cf_vals.min(), cf_vals.max()
        if cf_max > cf_min:
            cf_norm = (cf_vals - cf_min) / (cf_max - cf_min)
        else:
            cf_norm = np.zeros_like(cf_vals)

        # Normalize content scores
        ct_vals = np.array([ct_scores.get(r, 0.0) for r in all_items])
        ct_min, ct_max = ct_vals.min(), ct_vals.max()
        if ct_max > ct_min:
            ct_norm = (ct_vals - ct_min) / (ct_max - ct_min)
        else:
            ct_norm = np.zeros_like(ct_vals)

        # Combine
        hybrid = alpha * cf_norm + (1 - alpha) * ct_norm

        items_list = list(all_items)
        ranked = np.argsort(-hybrid)
        out = [(items_list[idx], float(hybrid[idx])) for idx in ranked[:K]]
        recs[u] = out
    return recs

# Use best C0 SVD model
svd_score_mat = c0_best["score_mat"]
svd_u2i = c0_best["u2i"]
svd_r2i = c0_best["r2i"]
svd_i2r = c0_best["i2r"]

c2_best = None
for alpha in tqdm(C2_ALPHA_VALUES, desc="C2 weighted hybrid tuning"):
    t0 = time.time()
    recs = weighted_hybrid_recs(alpha, svd_score_mat, svd_u2i, svd_i2r, best_tfidf_matrix, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"C2_a{int(alpha*100)}", f"Weighted hybrid alpha={alpha}", "C2", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"alpha_cf": alpha, "alpha_content": round(1-alpha, 2)})
    agg["notes"] = "Weighted hybrid: alpha*norm(CF) + (1-alpha)*norm(content)"
    tuning_rows.append(agg.copy())
    if (c2_best is None) or (agg["ndcg_at_k"] > c2_best["agg"]["ndcg_at_k"]):
        c2_best = {"agg": agg, "per_user": per_user, "recs": recs, "alpha": alpha}

metrics_rows.append(c2_best["agg"])
per_user_frames.append(c2_best["per_user"])
runtime_rows.append({"phase": "C2", "model_id": c2_best["agg"]["model_id"],
                      "runtime_seconds": c2_best["agg"]["runtime_seconds"],
                      "evaluated_users": c2_best["agg"]["evaluated_users"],
                      "seconds_per_user": c2_best["agg"]["seconds_per_user"]})

print(f"C2 best: {c2_best['agg']['model_id']}  NDCG@{K}: {c2_best['agg']['ndcg_at_k']:.6f}  alpha={c2_best['alpha']}")

## C3: Switching Hybrid

Smart strategy: check each user's training history count.
- If history >= threshold: use CF (SVD) recommendations
- If history < threshold: use content (TF-IDF) recommendations

This handles cold-start and sparse users better than a uniform approach.

In [ ]:
# ---- C3: Switching hybrid ----
def switching_hybrid_recs(threshold, score_mat_svd, u2i_svd, i2r_svd, tfidf_matrix, users_eval):
    recs = {}
    cf_count = 0
    ct_count = 0
    for u in users_eval:
        n_hist = user_train_count.get(u, 0)
        if n_hist >= threshold and u in u2i_svd:
            # Use CF
            seen = user_train_all_items.get(u, set())
            uidx = u2i_svd[u]
            s = score_mat_svd[uidx]
            idx_rank = np.argsort(-s)
            out = []
            for idx in idx_rank:
                item = i2r_svd[int(idx)]
                if item in seen:
                    continue
                out.append((item, float(s[idx])))
                if len(out) >= K:
                    break
            recs[u] = out if out else popularity_recs_for_users(pop_ranked, [u])[u]
            cf_count += 1
        else:
            # Use content
            from sklearn.metrics.pairwise import cosine_similarity
            liked = user_train_positive_items.get(u, set())
            liked_in_model = [recipe_id_to_idx[r] for r in liked if r in recipe_id_to_idx]
            if not liked_in_model:
                recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
                ct_count += 1
                continue
            user_vec = tfidf_matrix[liked_in_model].mean(axis=0)
            user_vec = np.asarray(user_vec).reshape(1, -1)
            sims = cosine_similarity(user_vec, tfidf_matrix).flatten()
            seen = user_train_all_items.get(u, set())
            idx_rank = np.argsort(-sims)
            out = []
            for idx in idx_rank:
                rid = int(recipe_ids_array[idx])
                if rid in seen:
                    continue
                out.append((rid, float(sims[idx])))
                if len(out) >= K:
                    break
            recs[u] = out if out else popularity_recs_for_users(pop_ranked, [u])[u]
            ct_count += 1
    print(f"  threshold={threshold}: CF users={cf_count}, content users={ct_count}")
    return recs

c3_best = None
for thr in tqdm(C3_THRESHOLD_VALUES, desc="C3 switching hybrid tuning"):
    t0 = time.time()
    recs = switching_hybrid_recs(thr, svd_score_mat, svd_u2i, svd_i2r, best_tfidf_matrix, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"C3_t{thr}", f"Switching hybrid threshold={thr}", "C3", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"switch_threshold": thr})
    agg["notes"] = "Switching hybrid: CF if history>=threshold, else content"
    tuning_rows.append(agg.copy())
    if (c3_best is None) or (agg["ndcg_at_k"] > c3_best["agg"]["ndcg_at_k"]):
        c3_best = {"agg": agg, "per_user": per_user, "recs": recs, "threshold": thr}

metrics_rows.append(c3_best["agg"])
per_user_frames.append(c3_best["per_user"])
runtime_rows.append({"phase": "C3", "model_id": c3_best["agg"]["model_id"],
                      "runtime_seconds": c3_best["agg"]["runtime_seconds"],
                      "evaluated_users": c3_best["agg"]["evaluated_users"],
                      "seconds_per_user": c3_best["agg"]["seconds_per_user"]})

print(f"C3 best: {c3_best['agg']['model_id']}  NDCG@{K}: {c3_best['agg']['ndcg_at_k']:.6f}  threshold={c3_best['threshold']}")

## C4: Reciprocal Rank Fusion (RRF) Hybrid

A rank-based fusion that does not depend on score scales:

```
RRF_score(item) = 1/(k + rank_cf) + 1/(k + rank_content)
```

where `k` is a smoothing constant (typically 60). This is robust because it only uses rank positions, not raw scores.

In [ ]:
# ---- C4: Reciprocal Rank Fusion ----
def rrf_hybrid_recs(rrf_k, score_mat_svd, u2i_svd, i2r_svd, tfidf_matrix, users_eval, n_candidates=200):
    """Reciprocal Rank Fusion of CF and content top-N lists."""
    recs = {}
    for u in users_eval:
        seen = user_train_all_items.get(u, set())

        # Get CF ranking
        cf_ranked = []
        if u in u2i_svd:
            uidx = u2i_svd[u]
            s = score_mat_svd[uidx]
            idx_rank = np.argsort(-s)
            for idx in idx_rank:
                item = i2r_svd[int(idx)]
                if item in seen:
                    continue
                cf_ranked.append(item)
                if len(cf_ranked) >= n_candidates:
                    break

        # Get content ranking
        ct_ranked = []
        liked = user_train_positive_items.get(u, set())
        liked_in_model = [recipe_id_to_idx[r] for r in liked if r in recipe_id_to_idx]
        if liked_in_model:
            from sklearn.metrics.pairwise import cosine_similarity
            user_vec = tfidf_matrix[liked_in_model].mean(axis=0)
            user_vec = np.asarray(user_vec).reshape(1, -1)
            sims = cosine_similarity(user_vec, tfidf_matrix).flatten()
            idx_rank = np.argsort(-sims)
            for idx in idx_rank:
                rid = int(recipe_ids_array[idx])
                if rid in seen:
                    continue
                ct_ranked.append(rid)
                if len(ct_ranked) >= n_candidates:
                    break

        if not cf_ranked and not ct_ranked:
            recs[u] = popularity_recs_for_users(pop_ranked, [u])[u]
            continue

        # Build rank maps
        cf_rank_map = {item: rank for rank, item in enumerate(cf_ranked, start=1)}
        ct_rank_map = {item: rank for rank, item in enumerate(ct_ranked, start=1)}
        default_rank = n_candidates + 1

        all_items = set(cf_ranked) | set(ct_ranked)
        rrf_scores = {}
        for item in all_items:
            r_cf = cf_rank_map.get(item, default_rank)
            r_ct = ct_rank_map.get(item, default_rank)
            rrf_scores[item] = 1.0 / (rrf_k + r_cf) + 1.0 / (rrf_k + r_ct)

        ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:K]
        recs[u] = ranked
    return recs

c4_best = None
for rk in tqdm(C4_RRF_K_VALUES, desc="C4 RRF tuning"):
    t0 = time.time()
    recs = rrf_hybrid_recs(rk, svd_score_mat, svd_u2i, svd_i2r, best_tfidf_matrix, eval_users)
    runtime = time.time() - t0
    agg, per_user = evaluate_model(f"C4_rrf{rk}", f"RRF hybrid k={rk}", "C4", split_name, recs, runtime)
    agg["parameters"] = json.dumps({"rrf_k": rk, "n_candidates": 200})
    agg["notes"] = "Reciprocal Rank Fusion of CF and content top-N lists"
    tuning_rows.append(agg.copy())
    if (c4_best is None) or (agg["ndcg_at_k"] > c4_best["agg"]["ndcg_at_k"]):
        c4_best = {"agg": agg, "per_user": per_user, "recs": recs}

metrics_rows.append(c4_best["agg"])
per_user_frames.append(c4_best["per_user"])
runtime_rows.append({"phase": "C4", "model_id": c4_best["agg"]["model_id"],
                      "runtime_seconds": c4_best["agg"]["runtime_seconds"],
                      "evaluated_users": c4_best["agg"]["evaluated_users"],
                      "seconds_per_user": c4_best["agg"]["seconds_per_user"]})

print(f"C4 best: {c4_best['agg']['model_id']}  NDCG@{K}: {c4_best['agg']['ndcg_at_k']:.6f}")

## Final Comparison and Export

In [ ]:
# ---- Final comparison ----
metrics_df = pd.DataFrame(metrics_rows)
tuning_df = pd.DataFrame(tuning_rows)
per_user_df = pd.concat(per_user_frames, ignore_index=True) if per_user_frames else pd.DataFrame()
runtime_df = pd.DataFrame(runtime_rows)

best_final = metrics_df.sort_values(["ndcg_at_k", "recall_at_k", "precision_at_k"], ascending=False).iloc[0]
final_model_id = best_final["model_id"]

# Select final recs
if final_model_id.startswith("C0"):
    final_recs = c0_best["recs"]
elif final_model_id.startswith("C1"):
    final_recs = c1_best["recs"]
elif final_model_id.startswith("C2"):
    final_recs = c2_best["recs"]
elif final_model_id.startswith("C3"):
    final_recs = c3_best["recs"]
else:
    final_recs = c4_best["recs"]

display(metrics_df[["model_id", "model_name", "precision_at_k", "recall_at_k",
                     "ndcg_at_k", "hit_at_k", "catalog_coverage_at_k", "runtime_seconds"]])
print("\nSelected final model:", final_model_id)

In [ ]:
# ---- Save results ----
prefix = "debug_" if DEBUG_MODE else ""

metrics_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_metrics.csv")
tuning_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_tuning_results.csv")
per_user_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_per_user_metrics.csv")
runtime_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_phase_runtime.csv")
config_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_config.json")
notes_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_model_notes.md")
examples_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_example_recommendations.csv")
top10_path = os.path.join(RESULTS_DIR, f"{prefix}version_c_top10_recommendations.csv")

metrics_df.to_csv(metrics_path, index=False)
tuning_df.to_csv(tuning_path, index=False)
per_user_df.to_csv(per_user_path, index=False)
runtime_df.to_csv(runtime_path, index=False)

config_payload = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "data_files": required_files,
    "relevant_item_definition": f"rating >= {POSITIVE_THRESHOLD}",
    "split": split_name,
    "k": K,
    "c0_component_values": C0_COMPONENT_VALUES,
    "c1_max_features_values": C1_MAX_FEATURES_VALUES,
    "c2_alpha_values": C2_ALPHA_VALUES,
    "c3_threshold_values": C3_THRESHOLD_VALUES,
    "c4_rrf_k_values": C4_RRF_K_VALUES,
    "debug_mode": DEBUG_MODE,
    "full_run": FULL_RUN,
    "selected_final_model": final_model_id,
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

with open(notes_path, "w", encoding="utf-8") as f:
    f.write("# Version C Model Notes\n\n")
    for _, row in metrics_df.iterrows():
        f.write(f"## {row['model_id']} - {row['model_name']}\n")
        f.write(f"- phase: {row['phase']}\n")
        f.write(f"- precision@{K}: {row['precision_at_k']:.6f}\n")
        f.write(f"- recall@{K}: {row['recall_at_k']:.6f}\n")
        f.write(f"- ndcg@{K}: {row['ndcg_at_k']:.6f}\n")
        f.write(f"- hit@{K}: {row['hit_at_k']:.6f}\n")
        f.write(f"- coverage@{K}: {row['catalog_coverage_at_k']:.6f}\n")
        f.write(f"- runtime_seconds: {row['runtime_seconds']:.4f}\n")
        f.write(f"- parameters: {row.get('parameters','{}')}\n")
        f.write(f"- notes: {row.get('notes','')}\n\n")

# Example recommendations
example_users = eval_users[:10]
example_rows = []
for u in example_users:
    hist = sorted(list(user_train_all_items.get(u, set())))[:50]
    rel = sorted(list(user_test_relevant_items.get(u, set())))
    rec_items = final_recs.get(u, [])
    rec_ids = [x[0] if isinstance(x, tuple) else x for x in rec_items]
    example_rows.append({
        "user_id": u, "model_id": final_model_id,
        "user_history_items": "|".join(map(str, hist)),
        "recommended_items": "|".join(map(str, rec_ids)),
        "relevant_test_items": "|".join(map(str, rel)),
        "explanation": "Hybrid recommendation combining CF and content signals.",
    })
pd.DataFrame(example_rows).to_csv(examples_path, index=False)

# Top-10 for all eval users
top_rows = []
for u in eval_users:
    rec_items = final_recs.get(u, [])
    for rk, rs in enumerate(rec_items[:K], start=1):
        rid, score = rs if isinstance(rs, tuple) else (rs, np.nan)
        top_rows.append({
            "user_id": u, "rank": rk, "recipe_id": rid,
            "score": float(score) if pd.notna(score) else np.nan,
            "model_id": final_model_id,
            "model_name": metrics_df.loc[metrics_df["model_id"] == final_model_id, "model_name"].iloc[0],
        })
pd.DataFrame(top_rows).to_csv(top10_path, index=False)

print("Results saved:")
for p in [metrics_path, tuning_path, per_user_path, runtime_path, config_path, notes_path, examples_path, top10_path]:
    print(f"  {p}")

In [ ]:
# ---- Visualization ----
plot_df = metrics_df.copy().sort_values("model_id")

# 1. Metric comparison bar chart
plt.figure(figsize=(10, 5))
x = np.arange(len(plot_df))
w = 0.25
plt.bar(x - w, plot_df["precision_at_k"], w, label="precision@10")
plt.bar(x, plot_df["recall_at_k"], w, label="recall@10")
plt.bar(x + w, plot_df["ndcg_at_k"], w, label="ndcg@10")
plt.xticks(x, plot_df["model_id"], rotation=20)
plt.title("Version C Model Metrics at K=10")
plt.ylabel("metric value")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_model_metrics_at_10.png"), dpi=150)
plt.show()
plt.close()

# 2. Coverage bar chart
plt.figure(figsize=(8, 4.5))
plt.bar(plot_df["model_id"], plot_df["catalog_coverage_at_k"])
plt.title("Version C Catalog Coverage at K=10")
plt.ylabel("catalog_coverage")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_catalog_coverage_at_10.png"), dpi=150)
plt.show()
plt.close()

# 3. Quality vs runtime scatter
plt.figure(figsize=(8, 5))
plt.scatter(plot_df["seconds_per_user"], plot_df["ndcg_at_k"])
for _, r in plot_df.iterrows():
    plt.annotate(r["model_id"], (r["seconds_per_user"], r["ndcg_at_k"]),
                 xytext=(4, 4), textcoords="offset points", fontsize=8)
plt.title("Version C Quality vs Runtime at K=10")
plt.xlabel("seconds_per_user")
plt.ylabel("ndcg@10")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_quality_vs_runtime_at_10.png"), dpi=150)
plt.show()
plt.close()

# 4. Hit rate
plt.figure(figsize=(8, 4.5))
plt.bar(plot_df["model_id"], plot_df["hit_at_k"])
plt.title("Version C Hit Rate at K=10")
plt.ylabel("hit@10")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_hit_rate_at_10.png"), dpi=150)
plt.show()
plt.close()

# 5. Phase runtime
plt.figure(figsize=(8, 4.5))
rt = runtime_df.sort_values("phase")
plt.bar(rt["phase"], rt["runtime_seconds"])
plt.title("Version C Phase Runtime")
plt.ylabel("runtime_seconds")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_phase_runtime.png"), dpi=150)
plt.show()
plt.close()

# 6. Per-user NDCG boxplot
if not per_user_df.empty:
    box_models = sorted(per_user_df["model_id"].unique())
    box_data = [per_user_df.loc[per_user_df["model_id"] == m, "ndcg_at_k"].values for m in box_models]
    plt.figure(figsize=(10, 5))
    plt.boxplot(box_data, labels=box_models, showfliers=False)
    plt.xticks(rotation=25)
    plt.title("Version C Per-user NDCG@10 Distribution")
    plt.ylabel("ndcg@10")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_per_user_ndcg_boxplot.png"), dpi=150)
    plt.show()
    plt.close()

# 7. Hit rank distribution for final model
rank_hits = []
for u in eval_users:
    rel = user_test_relevant_items.get(u, set())
    rec_items = final_recs.get(u, [])
    rec_ids = [x[0] if isinstance(x, tuple) else x for x in rec_items][:K]
    for rnk, rid in enumerate(rec_ids, start=1):
        if rid in rel:
            rank_hits.append(rnk)
            break
if rank_hits:
    plt.figure(figsize=(7, 4.5))
    bins = np.arange(1, K + 2) - 0.5
    plt.hist(rank_hits, bins=bins, rwidth=0.9)
    plt.xticks(range(1, K + 1))
    plt.title("Version C Hit Rank Distribution at K=10")
    plt.xlabel("hit rank")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_hit_rank_distribution_at_10.png"), dpi=150)
    plt.show()
    plt.close()

print("All figures saved to:", FIGURES_DIR)

In [ ]:
# ---- Cross-version comparison (A vs B vs C) ----
print("="*60)
print("CROSS-VERSION COMPARISON")
print("="*60)
print()

# Load A and B metrics if available
cross_rows = []

a_path = "../../Version_A/Results/version_a_metrics.csv"
if os.path.isfile(a_path):
    a_df = pd.read_csv(a_path)
    # Best A model by NDCG
    best_a = a_df.sort_values("ndcg_at_k", ascending=False).iloc[0]
    cross_rows.append({"version": "A", "model_id": best_a["model_id"],
                       "model_name": best_a.get("model_name", ""),
                       "ndcg_at_k": best_a["ndcg_at_k"],
                       "recall_at_k": best_a["recall_at_k"],
                       "precision_at_k": best_a["precision_at_k"],
                       "catalog_coverage_at_k": best_a["catalog_coverage_at_k"]})
    print("Version A best loaded:", best_a["model_id"])

b_path = "../../Version_B/Results/version_b_metrics.csv"
if os.path.isfile(b_path):
    b_df = pd.read_csv(b_path)
    best_b = b_df.sort_values("ndcg_at_k", ascending=False).iloc[0]
    cross_rows.append({"version": "B", "model_id": best_b["model_id"],
                       "model_name": best_b.get("model_name", ""),
                       "ndcg_at_k": best_b["ndcg_at_k"],
                       "recall_at_k": best_b["recall_at_k"],
                       "precision_at_k": best_b["precision_at_k"],
                       "catalog_coverage_at_k": best_b["catalog_coverage_at_k"]})
    print("Version B best loaded:", best_b["model_id"])

# Add best C
cross_rows.append({"version": "C", "model_id": final_model_id,
                   "model_name": best_final.get("model_name", ""),
                   "ndcg_at_k": best_final["ndcg_at_k"],
                   "recall_at_k": best_final["recall_at_k"],
                   "precision_at_k": best_final["precision_at_k"],
                   "catalog_coverage_at_k": best_final["catalog_coverage_at_k"]})

cross_df = pd.DataFrame(cross_rows)
display(cross_df)

cross_df.to_csv(os.path.join(RESULTS_DIR, f"{prefix}version_c_cross_comparison.csv"), index=False)

if len(cross_df) > 1:
    plt.figure(figsize=(8, 5))
    x = np.arange(len(cross_df))
    w = 0.2
    plt.bar(x - w, cross_df["precision_at_k"], w, label="precision@10")
    plt.bar(x, cross_df["recall_at_k"], w, label="recall@10")
    plt.bar(x + w, cross_df["ndcg_at_k"], w, label="ndcg@10")
    labels = [f"{r['version']}:{r['model_id']}" for _, r in cross_df.iterrows()]
    plt.xticks(x, labels, rotation=20)
    plt.title("Cross-Version Best Model Comparison")
    plt.ylabel("metric value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{prefix}version_c_cross_version_comparison.png"), dpi=150)
    plt.show()
    plt.close()

print("\nDone. Final model:", final_model_id)